<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-5-agents/Module_5_Session_1_Tool_Use.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5 — Session 1: Tool Use & Function Calling

## What are we building?
A Swiggy support agent that can *call real Python functions* to check order status, calculate refunds, and answer customer queries — instead of just guessing.

## Why does this matter?
Tool use is what separates a chatbot from an agent. The LLM decides *when* to call a tool and *what to send* — we just define the tools and handle the results.

## What we will learn:
- How to define a tool (Python function) and describe it to the LLM
- How the LLM decides to call a tool (the tool_calls response)
- How to execute the tool and send results back
- Full loop: User query → LLM picks tool → We run it → LLM answers

## Model: openai/gpt-oss-20b on Groq
## Platform: Google Colab
## API: Groq (same API key as always)

## Step 1: Install and Setup
Same groq library we have used since Module 0. No new libraries this session.

In [ ]:
# Install groq library (same as always)
!pip install groq -q

# Import libraries we need
from groq import Groq          # to talk to Groq API
import os                      # to access environment variables
import json                    # NEW: to parse tool arguments the LLM sends back

# Setup Groq client using your API key from Colab Secrets
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = Groq()

# Our model for this entire module
MODEL = "openai/gpt-oss-20b"

print("Setup complete. Model:", MODEL)

## Step 2: Define Our Tools (Python Functions)
These are real Python functions that do actual work.
The LLM cannot run these — only WE can. The LLM just tells us which one to call and with what arguments.

In [ ]:
# --- TOOL 1: Check Swiggy Order Status ---
# Simulates looking up an order in a database
def check_order_status(order_id):
    # Fake order database (in real Swiggy this would be a SQL query)
    orders = {
        "ORD001": {"status": "Out for delivery", "eta": "10 minutes", "restaurant": "Burger King"},
        "ORD002": {"status": "Delivered", "eta": None, "restaurant": "Pizza Hut"},
        "ORD003": {"status": "Preparing", "eta": "25 minutes", "restaurant": "McDonald's"},
    }

    # Look up order, return error message if not found
    order = orders.get(order_id)
    if order:
        return f"Order {order_id} from {order['restaurant']}: {order['status']}. ETA: {order['eta']}"
    else:
        return f"Order {order_id} not found in system."

# --- TOOL 2: Calculate Refund ---
# Simulates calculating refund amount based on order value and reason
def calculate_refund(order_id, reason):
    # Fake order values
    order_values = {
        "ORD001": 450,
        "ORD002": 320,
        "ORD003": 180,
    }

    amount = order_values.get(order_id, 0)

    # Refund policy: wrong item = 100%, late delivery = 30%
    if reason == "wrong_item":
        refund = amount * 1.0
    elif reason == "late_delivery":
        refund = amount * 0.3
    else:
        refund = 0

    return f"Refund for {order_id}: ₹{refund} approved for reason: {reason}"

# Quick test to confirm functions work independently
print(check_order_status("ORD001"))
print(calculate_refund("ORD002", "wrong_item"))

## Step 3: Describe Tools to the LLM
The LLM cannot see our Python code. We have to describe each tool
in a special JSON format so the LLM knows — what is the tool called,
what does it do, and what arguments does it need.

In [ ]:
# This is the "menu of tools" we hand to the LLM
# The LLM reads these descriptions to decide which tool to call

tools = [
    {
        "type": "function",          # always "function" for custom tools
        "function": {
            "name": "check_order_status",         # must match your Python function name exactly
            "description": "Check the current status of a Swiggy order using the order ID. Use this when customer asks where their order is or what the delivery status is.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The Swiggy order ID, for example ORD001"
                    }
                },
                "required": ["order_id"]          # LLM must provide this argument
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_refund",
            "description": "Calculate the refund amount for a Swiggy order. Use this when customer complaints about wrong item delivered or late delivery and wants a refund.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The Swiggy order ID"
                    },
                    "reason": {
                        "type": "string",
                        "description": "Reason for refund. Must be either 'wrong_item' or 'late_delivery'",
                        "enum": ["wrong_item", "late_delivery"]   # LLM can only pick from these values
                    }
                },
                "required": ["order_id", "reason"]
            }
        }
    }
]

print("Tools defined:", len(tools))
print("Tool 1:", tools[0]["function"]["name"])
print("Tool 2:", tools[1]["function"]["name"])

## Step 4: First LLM Call — Let the LLM Decide Which Tool to Call
We send the customer query + tool descriptions to the LLM.
The LLM does NOT answer the question directly.
Instead it says — "I need to call this tool with these arguments."
We call this the tool_calls response.

In [ ]:
# Customer query coming into Swiggy support
customer_query = "Hi, I placed order ORD001 but I have no idea where it is. Can you help?"

# First LLM call — we give it the query AND the tools menu
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful Swiggy customer support agent."},
        {"role": "user", "content": customer_query}
    ],
    tools=tools,           # hand the LLM our tool descriptions
    tool_choice="auto"     # let LLM decide whether to call a tool or just reply
)

# Let us see what the LLM decided to do
message = response.choices[0].message

print("LLM decision:")
print("Content:", message.content)
print("Tool calls:", message.tool_calls)

## Step 5: We Run the Tool and Send Results Back
The LLM told us which tool to call and with what arguments.
Now we execute the actual Python function and send the result
back to the LLM so it can give a final answer to the customer.

In [ ]:
# Extract the tool call details from LLM response
tool_call = message.tool_calls[0]                          # get first tool call
tool_name = tool_call.function.name                        # which function to run
tool_args = json.loads(tool_call.function.arguments)       # convert JSON string → Python dict

print("Tool to run:", tool_name)
print("Arguments:", tool_args)

# Run the actual Python function dynamically
# We use a dispatch dictionary — maps tool name to actual function
available_tools = {
    "check_order_status": check_order_status,
    "calculate_refund": calculate_refund
}

# Call whichever function the LLM requested
tool_result = available_tools[tool_name](**tool_args)      # ** unpacks dict as keyword arguments
print("Tool result:", tool_result)

## Step 6: Second LLM Call — Generate Final Answer
We now send the full conversation back to the LLM:
- Original customer query
- LLM's tool call request  
- Our tool result
The LLM reads all of this and writes a proper customer support reply.

In [ ]:
# Build the full conversation history to send back to LLM
# This is the complete story: query → tool request → tool result → final answer

final_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        # 1. System instruction
        {"role": "system", "content": "You are a helpful Swiggy customer support agent."},

        # 2. Original customer query
        {"role": "user", "content": customer_query},

        # 3. LLM's own tool call request (we send it back so LLM remembers what it asked)
        {"role": "assistant", "content": None, "tool_calls": [
            {
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_name,
                    "arguments": tool_call.function.arguments
                }
            }
        ]},

        # 4. The result our Python function returned
        {"role": "tool",                        # special role — this is a tool result
         "tool_call_id": tool_call.id,          # must match the id from step above
         "content": tool_result}                # actual result from our function
    ]
)

# Final answer for the customer
final_answer = final_response.choices[0].message.content
print("Final answer to customer:")
print(final_answer)

## Step 7: Putting It All Together — Reusable Agent Function
Now we refactor everything into one clean reusable function.
DRY principle — one function handles any customer query, any tool.

In [ ]:
def run_swiggy_agent(customer_query):
    """
    Full tool-use loop in one function:
    1. Send query to LLM with tools
    2. If LLM wants a tool — run it
    3. Send result back to LLM
    4. Return final answer
    """

    print(f"Customer: {customer_query}")
    print("-" * 50)

    # --- STEP 1: First LLM call ---
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful Swiggy customer support agent."},
            {"role": "user", "content": customer_query}
        ],
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # --- STEP 2: Check if LLM wants to call a tool ---
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        print(f"Tool called: {tool_name}")
        print(f"Arguments: {tool_args}")

        # --- STEP 3: Run the tool ---
        available_tools = {
            "check_order_status": check_order_status,
            "calculate_refund": calculate_refund
        }
        tool_result = available_tools[tool_name](**tool_args)
        print(f"Tool result: {tool_result}")
        print("-" * 50)

        # --- STEP 4: Second LLM call with tool result ---
        final_response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a helpful Swiggy customer support agent."},
                {"role": "user", "content": customer_query},
                {"role": "assistant", "content": None, "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_name,
                            "arguments": tool_call.function.arguments
                        }
                    }
                ]},
                {"role": "tool",
                 "tool_call_id": tool_call.id,
                 "content": tool_result}
            ]
        )
        return final_response.choices[0].message.content

    else:
        # LLM answered directly without needing a tool
        print("No tool needed — LLM answered directly")
        return message.content

# --- Test with 3 different customer queries ---
queries = [
    "What is the status of my order ORD003?",
    "I received the wrong item in order ORD002, I want a refund",
    "What are your customer support hours?"     # no tool needed for this one
]

for query in queries:
    answer = run_swiggy_agent(query)
    print(f"Agent: {answer}")
    print("=" * 60)

## Summary

### What we built:
A Swiggy support agent that uses real Python functions to answer
customer queries — instead of guessing.

### The full tool-use loop:
1. Customer sends query
2. LLM reads query + tool descriptions → decides which tool to call
3. We run the Python function with LLM's arguments
4. We send result back to LLM
5. LLM writes final human response

### Key concepts:
- Tool description = prompt for the tool. Bad description = wrong tool called.
- LLM never runs code — it only requests. We execute.
- tool_call.id links requests to results like a ticket number.
- tool_choice="auto" lets LLM decide whether a tool is needed at all.
- ** unpacks a dictionary into keyword arguments.

### AWS Equivalent:
| This session | AWS |
|---|---|
| Tool definition (JSON schema) | AWS Lambda function + API Gateway schema |
| Tool dispatcher (available_tools dict) | AWS Step Functions choice state |
| Tool result sent back to LLM | Lambda response → Bedrock Agent |
| Full agent loop | Amazon Bedrock Agents (does all this automatically) |

### Next session:
Module 5 Session 2 — The Agent Loop (ReAct from scratch)
We will build the Reason → Act → Observe loop in raw Python
so you understand what LangGraph and Bedrock Agents hide under the hood.